In [3]:
import numpy as np
import pandas as pd

In [4]:
data_claim = pd.read_csv('data/Data_Klaim.csv')
data_polis = pd.read_csv('data/Data_Polis.csv')

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Weekly
weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# ==========================================
# 2. DLINEAR DECOMPOSITION & FEATURE ENGINEERING
# ==========================================
print("2. Performing DLinear Decomposition...")

window_size = 4 # 4-week moving average for Trend

targets = ['Frequency', 'Total_Claim']

for col in targets:
    # 1. Extract Trend (Moving Average)
    weekly_df[f'{col}_Trend'] = weekly_df[col].rolling(window=window_size).mean()
    
    # 2. Extract Remainder (Raw - Trend)
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    # 3. Create Lags for the Linear Layers (Lookback = 4 weeks)
    for lag in range(1, 5):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN DLINEAR LINEAR LAYERS
# ==========================================
print("3. Training Dual Linear Layers...")

# We use Ridge (Regularized Linear Regression) to prevent wild coefficient spikes
model_params = {'alpha': 1.0}

# --- FREQUENCY MODELS ---
features_freq_trend = [f'Frequency_Trend_Lag{i}' for i in range(1, 5)]
model_freq_trend = Ridge(**model_params)
model_freq_trend.fit(train_df[features_freq_trend], train_df['Frequency_Trend'])

features_freq_remain = [f'Frequency_Remain_Lag{i}' for i in range(1, 5)]
model_freq_remain = Ridge(**model_params)
model_freq_remain.fit(train_df[features_freq_remain], train_df['Frequency_Remain'])

# --- TOTAL CLAIM MODELS ---
features_tot_trend = [f'Total_Claim_Trend_Lag{i}' for i in range(1, 5)]
model_tot_trend = Ridge(**model_params)
model_tot_trend.fit(train_df[features_tot_trend], train_df['Total_Claim_Trend'])

features_tot_remain = [f'Total_Claim_Remain_Lag{i}' for i in range(1, 5)]
model_tot_remain = Ridge(**model_params)
model_tot_remain.fit(train_df[features_tot_remain], train_df['Total_Claim_Remain'])

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING
# ==========================================
print("4. Forecasting Future Weeks recursively...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    # Get the last 4 known rows to build our lag features
    last_4 = current_history.tail(4)
    
    # --- FREQUENCY PREDICTION ---
    row_freq_trend = pd.DataFrame([{f'Frequency_Trend_Lag{i}': last_4['Frequency_Trend'].iloc[-i] for i in range(1, 5)}])
    row_freq_remain = pd.DataFrame([{f'Frequency_Remain_Lag{i}': last_4['Frequency_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_f_trend = model_freq_trend.predict(row_freq_trend)[0]
    pred_f_remain = model_freq_remain.predict(row_freq_remain)[0]
    
    # DLinear Recombination: Output = Trend + Remainder
    pred_freq = pred_f_trend + pred_f_remain
    pred_freq = max(0, pred_freq) # Safety cap
    
    # --- TOTAL CLAIM PREDICTION ---
    row_tot_trend = pd.DataFrame([{f'Total_Claim_Trend_Lag{i}': last_4['Total_Claim_Trend'].iloc[-i] for i in range(1, 5)}])
    row_tot_remain = pd.DataFrame([{f'Total_Claim_Remain_Lag{i}': last_4['Total_Claim_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_t_trend = model_tot_trend.predict(row_tot_trend)[0]
    pred_t_remain = model_tot_remain.predict(row_tot_remain)[0]
    
    pred_total = pred_t_trend + pred_t_remain
    pred_total = max(0, pred_total)
    
    # Store Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update history for the next loop so the Moving Average updates correctly
    new_row_data = {'Week_End_Date': week_end, 'Frequency': pred_freq, 'Total_Claim': pred_total}
    
    # Calculate the new Trend and Remainder based on this new prediction
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    # Update Decomposition mathematically
    for col in targets:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].rolling(window=window_size).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# Roll up explicitly selecting numeric columns to prevent datetime sum errors
cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

# Filter for Target Window (Aug 2025 - Dec 2025)
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

# Mathematical Alignments
final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
# Derived Severity to prevent compounding evaluation errors
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_dlinear.csv', index=False)

print("\n--- FINAL FORECAST (DLinear Architecture) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_dlinear.csv' saved! Time to drop it in the leaderboard!")

1. Processing Raw Data to Weekly Level...
2. Performing DLinear Decomposition...
3. Training Dual Linear Layers...
4. Forecasting Future Weeks recursively...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (DLinear Architecture) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        220  5.054340e+07  1.111955e+10
1      2025-09        237  5.233406e+07  1.240317e+10
2      2025-10        247  5.299879e+07  1.309070e+10
3      2025-11        239  5.341499e+07  1.276618e+10
4      2025-12        248  5.334536e+07  1.322965e+10

File 'submission_dlinear.csv' saved! Time to drop it in the leaderboard!
